In [ ]:
# ============================================================
# CELL 1: Install libraries + imports
# Run this first. Wait for it to finish before moving to Cell 2.
# ============================================================

# Install one library that Colab doesn't have by default
!pip install -q plotly

# --- Standard imports ---
import pandas as pd
import sqlite3
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings("ignore")

# --- Display settings ---
pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", "{:,.2f}".format)

print("✅ All libraries loaded. Move to Cell 2.")

✅ All libraries loaded. Move to Cell 2.


In [ ]:
# ============================================================
# CELL 2: Upload & Load CSV
# ============================================================
from google.colab import files

print("A file picker will appear. Select your perfume_prices CSV file.")
uploaded = files.upload()
# files.upload() opens a file picker in your browser
# Navigate to your project folder and select the CSV

import io
filename = list(uploaded.keys())[0]
# list(uploaded.keys())[0] = gets the name of the file you uploaded

df_raw = pd.read_csv(io.BytesIO(uploaded[filename]))
# BytesIO converts the uploaded file bytes into something pandas can read

print(f"✅ Loaded {len(df_raw)} rows, {df_raw.shape[1]} columns.")
print(df_raw.head())

A file picker will appear. Select your perfume_prices CSV file.


Saving perfume_prices_20260424_0433.csv to perfume_prices_20260424_0433 (2).csv
✅ Loaded 166 rows, 10 columns.
    platform search_term                                   product_name  \
0  Amazon.in     perfume  Bella Vita Luxury Perfumes Gift Set for Women   
1  Amazon.in     perfume      Blabliblu Love Drunk Long Lasting Perfume   
2  Amazon.in     perfume        Engage Luxury Perfume Gift Pack for Men   
3  Amazon.in     perfume        Nykaa Women's Cherry Bomb Eau De Parfum   
4  Amazon.in     perfume               Beardo Godfather Perfume for Men   

       brand  price_inr  mrp_inr  discount_pct  rating  review_count  \
0  Amazon.in     549.00      NaN           NaN    4.70      2,200.00   
1  Amazon.in     199.00      NaN           NaN     NaN           NaN   
2  Amazon.in     399.00      NaN           NaN    5.00          1.00   
3  amazon.in     869.00      NaN           NaN     NaN           NaN   
4  Amazon.in     159.00      NaN           NaN    4.30      1,200.00   

     

In [25]:
# ============================================================
# CELL 3: Clean the Data
# ============================================================

df = df_raw.copy()
# .copy() = work on a copy so original is never modified

# 1. Strip whitespace from text columns
for col in ["platform", "product_name", "brand"]:
    df[col] = df[col].astype(str).str.strip()

# 2. Convert price columns to numeric (force errors to NaN)
df["price_inr"] = pd.to_numeric(df["price_inr"], errors="coerce")
df["mrp_inr"]   = pd.to_numeric(df["mrp_inr"],   errors="coerce")

# 3. Drop rows with no price AND no product name — useless rows
df = df.dropna(subset=["product_name"])
df = df[df["product_name"] != "N/A"]

# 4. Add price tier column — segments products like a business analyst would
def price_tier(price):
    if pd.isna(price):       return "Unknown"
    elif price < 500:        return "Budget"
    elif price < 2000:       return "Mid-range"
    elif price < 6000:       return "Premium"
    else:                    return "Luxury"

df["price_tier"] = df["price_inr"].apply(price_tier)

# Remove junk Nykaa category pages
junk_keywords = ["buy ", "online", "at best price", "at amazing"]
mask = df["product_name"].str.lower().apply(
    lambda x: any(k in x for k in junk_keywords)
)
df = df[~mask]


# 5. Clean up rating column — extract just the number
df["rating"] = pd.to_numeric(
    df["rating"].astype(str).str.extract(r"(\d+\.?\d*)")[0],
    errors="coerce"
)

# 6. Parse scraped_at as datetime
df["scraped_at"] = pd.to_datetime(df["scraped_at"], errors="coerce", utc=True)

print(f"✅ Clean dataset: {len(df)} rows")
print(f"\nPlatform breakdown:\n{df['platform'].value_counts()}")
print(f"\nPrice tier breakdown:\n{df['price_tier'].value_counts()}")
print(f"\nMissing prices: {df['price_inr'].isna().sum()} rows")

from google.colab import files
df.to_csv("perfume_prices_clean.csv", index=False, encoding="utf-8-sig")
files.download("perfume_prices_clean.csv")


✅ Clean dataset: 110 rows

Platform breakdown:
platform
Amazon.in    47
Flipkart     32
Nykaa        31
Name: count, dtype: int64

Price tier breakdown:
price_tier
Budget       44
Unknown      39
Mid-range    17
Luxury        5
Premium       5
Name: count, dtype: int64

Missing prices: 39 rows


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# CELL 4: Load into SQLite (your in-notebook warehouse)
# SQLite = a lightweight database that lives inside your notebook.
# Think of it as BigQuery but running locally for free.
# ============================================================

conn = sqlite3.connect(":memory:")
# ":memory:" = database lives in RAM, gone when notebook closes
# Perfect for analysis — no files needed

df.to_sql("perfume_prices", conn, if_exists="replace", index=False)
# Loads the entire DataFrame into a SQL table called "perfume_prices"

def sql(query):
    """Helper: run any SQL query and return a DataFrame."""
    return pd.read_sql_query(query, conn)

print("✅ Data loaded into SQLite. You can now run SQL queries.")
print(sql("SELECT platform, COUNT(*) as products FROM perfume_prices GROUP BY platform"))


✅ Data loaded into SQLite. You can now run SQL queries.
    platform  products
0  Amazon.in        47
1   Flipkart        49
2      Nykaa        70


In [ ]:

# ============================================================
# CELL 5: ANALYSIS 1 — Platform Price Comparison
# Business question: Which platform is cheapest for perfume?
# ============================================================

platform_stats = sql("""
    SELECT
        platform,
        COUNT(*)                        AS total_products,
        ROUND(AVG(price_inr), 0)        AS avg_price,
        ROUND(MIN(price_inr), 0)        AS min_price,
        ROUND(MAX(price_inr), 0)        AS max_price,
        ROUND(AVG(discount_pct), 1)     AS avg_discount_pct
    FROM perfume_prices
    WHERE price_inr IS NOT NULL
    GROUP BY platform
    ORDER BY avg_price
""")

print("=== PLATFORM PRICE COMPARISON ===")
print(platform_stats.to_string(index=False))

fig1 = px.bar(
    platform_stats,
    x="platform",
    y="avg_price",
    color="platform",
    text="avg_price",
    title="Average Perfume Price by Platform (₹)",
    labels={"avg_price": "Avg Price (₹)", "platform": "Platform"},
    color_discrete_map={
        "Amazon.in": "#FF9900",
        "Flipkart":  "#2874F0",
        "Nykaa":     "#FC2779"
    }
)
fig1.update_traces(texttemplate="₹%{text:,.0f}", textposition="outside")
fig1.update_layout(showlegend=False, plot_bgcolor="white")
fig1.show()



=== PLATFORM PRICE COMPARISON ===
 platform  total_products  avg_price  min_price  max_price avg_discount_pct
 Flipkart              17     759.00     140.00   2,322.00             None
Amazon.in              47   1,669.00     149.00  26,999.00             None
    Nykaa              27   4,707.00     349.00  10,500.00             None


In [ ]:

# ============================================================
# CELL 6: ANALYSIS 2 — Price Tier Distribution per Platform
# Business question: Does each platform target different customer segments?
# ============================================================

tier_dist = sql("""
    SELECT
        platform,
        price_tier,
        COUNT(*) AS product_count
    FROM perfume_prices
    WHERE price_tier != 'Unknown'
    GROUP BY platform, price_tier
    ORDER BY platform, price_tier
""")

print("=== PRICE TIER DISTRIBUTION ===")
print(tier_dist.to_string(index=False))

fig2 = px.bar(
    tier_dist,
    x="platform",
    y="product_count",
    color="price_tier",
    barmode="group",
    title="Price Tier Distribution by Platform",
    labels={"product_count": "Number of Products", "price_tier": "Tier"},
    category_orders={"price_tier": ["Budget", "Mid-range", "Premium", "Luxury"]},
    color_discrete_map={
        "Budget":    "#2ecc71",
        "Mid-range": "#3498db",
        "Premium":   "#9b59b6",
        "Luxury":    "#e74c3c"
    }
)
fig2.update_layout(plot_bgcolor="white")
fig2.show()



=== PRICE TIER DISTRIBUTION ===
 platform price_tier  product_count
Amazon.in     Budget             35
Amazon.in     Luxury              4
Amazon.in  Mid-range              6
Amazon.in    Premium              2
 Flipkart     Budget              9
 Flipkart  Mid-range              6
 Flipkart    Premium              2
    Nykaa     Budget              2
    Nykaa     Luxury             10
    Nykaa  Mid-range             12
    Nykaa    Premium              3


In [ ]:

# ============================================================
# CELL 7: ANALYSIS 3 — Top 15 Most Expensive Products
# Business question: What are the luxury anchors in this market?
# ============================================================

top_expensive = sql("""
    SELECT
        platform,
        product_name,
        price_inr,
        price_tier
    FROM perfume_prices
    WHERE price_inr IS NOT NULL
    ORDER BY price_inr DESC
    LIMIT 15
""")

print("=== TOP 15 MOST EXPENSIVE PRODUCTS ===")
print(top_expensive.to_string(index=False))

fig3 = px.bar(
    top_expensive,
    x="price_inr",
    y="product_name",
    color="platform",
    orientation="h",
    title="Top 15 Most Expensive Perfumes (₹)",
    labels={"price_inr": "Price (₹)", "product_name": "Product"},
    color_discrete_map={
        "Amazon.in": "#FF9900",
        "Flipkart":  "#2874F0",
        "Nykaa":     "#FC2779"
    }
)
fig3.update_layout(yaxis={"categoryorder": "total ascending"}, plot_bgcolor="white")
fig3.show()



=== TOP 15 MOST EXPENSIVE PRODUCTS ===
 platform                                                       product_name  price_inr price_tier
Amazon.in                                            Parfums De Marly Delina  26,999.00     Luxury
Amazon.in                               Dior Sauvage Eau De Toilette For Men  14,500.00     Luxury
    Nykaa                          Buy Edp Online In India At Amazing Offers  10,500.00     Luxury
    Nykaa                      Moody Premium Perfume Gift Set Of 4 For Women  10,500.00     Luxury
    Nykaa                  Buy Lancome La Vie Est Belle Eau De Parfum Online  10,500.00     Luxury
    Nykaa                       Buy Fragrance Online at Best Prices in India  10,500.00     Luxury
    Nykaa                 Buy Best Perfume For Women Online In India? At ...  10,500.00     Luxury
    Nykaa                                              Buy Fragrances Online  10,500.00     Luxury
    Nykaa                   Buy Luxury Perfume for Men Online at Best 

In [ ]:

# ============================================================
# CELL 8: ANALYSIS 4 — Budget Products (under ₹500)
# Business question: Which platform has the most affordable options?
# ============================================================

budget = sql("""
    SELECT
        platform,
        COUNT(*) AS budget_products,
        ROUND(AVG(price_inr), 0) AS avg_budget_price
    FROM perfume_prices
    WHERE price_inr < 500
    GROUP BY platform
    ORDER BY budget_products DESC
""")

print("=== BUDGET PRODUCTS (under ₹500) ===")
print(budget.to_string(index=False))

fig4 = px.pie(
    budget,
    names="platform",
    values="budget_products",
    title="Share of Budget Perfumes (under ₹500) by Platform",
    color="platform",
    color_discrete_map={
        "Amazon.in": "#FF9900",
        "Flipkart":  "#2874F0",
        "Nykaa":     "#FC2779"
    }
)
fig4.show()


=== BUDGET PRODUCTS (under ₹500) ===
 platform  budget_products  avg_budget_price
Amazon.in               35            282.00
 Flipkart                9            286.00
    Nykaa                2            349.00


In [ ]:

# ============================================================
# CELL 9: ANALYSIS 5 — Price Distribution (Box Plot)
# Business question: How spread out are prices on each platform?
# ============================================================

df_priced = df[df["price_inr"].notna()].copy()

fig5 = px.box(
    df_priced,
    x="platform",
    y="price_inr",
    color="platform",
    title="Price Distribution by Platform (₹)",
    labels={"price_inr": "Price (₹)", "platform": "Platform"},
    color_discrete_map={
        "Amazon.in": "#FF9900",
        "Flipkart":  "#2874F0",
        "Nykaa":     "#FC2779"
    }
)
fig5.update_layout(showlegend=False, plot_bgcolor="white", yaxis_type="log")
# yaxis_type="log" = log scale because prices range from ₹99 to ₹10,000+
fig5.show()



In [ ]:

# ============================================================
# CELL 10: ANALYSIS 6 — Search Term Performance
# Business question: Which search term yields the most products?
# ============================================================

term_perf = sql("""
    SELECT
        search_term,
        platform,
        COUNT(*)                    AS products_found,
        ROUND(AVG(price_inr), 0)   AS avg_price
    FROM perfume_prices
    WHERE price_inr IS NOT NULL
    GROUP BY search_term, platform
    ORDER BY search_term, platform
""")

print("=== SEARCH TERM PERFORMANCE ===")
print(term_perf.to_string(index=False))

fig6 = px.bar(
    term_perf,
    x="search_term",
    y="products_found",
    color="platform",
    barmode="group",
    title="Products Found per Search Term by Platform",
    labels={"products_found": "Products Found", "search_term": "Search Term"},
    color_discrete_map={
        "Amazon.in": "#FF9900",
        "Flipkart":  "#2874F0",
        "Nykaa":     "#FC2779"
    }
)
fig6.update_layout(plot_bgcolor="white")
fig6.show()



=== SEARCH TERM PERFORMANCE ===
  search_term  platform  products_found  avg_price
        attar Amazon.in              13     470.00
        attar  Flipkart               5     905.00
        attar     Nykaa               3     600.00
eau de parfum Amazon.in              10   3,994.00
eau de parfum  Flipkart               5   1,161.00
eau de parfum     Nykaa               9   7,261.00
    fragrance Amazon.in               8   2,584.00
    fragrance  Flipkart               2     227.00
    fragrance     Nykaa              10   4,229.00
      perfume Amazon.in              16     733.00
      perfume  Flipkart               5     423.00
      perfume     Nykaa               5   3,529.00


In [ ]:

# ============================================================
# CELL 11: BUSINESS SUMMARY — what you say in an interview
# ============================================================

summary = sql("""
    SELECT
        platform,
        COUNT(*)                        AS total_listings,
        ROUND(AVG(price_inr), 0)        AS avg_price_inr,
        ROUND(MIN(price_inr), 0)        AS min_price_inr,
        ROUND(MAX(price_inr), 0)        AS max_price_inr,
        SUM(CASE WHEN price_tier='Budget'    THEN 1 ELSE 0 END) AS budget_count,
        SUM(CASE WHEN price_tier='Mid-range' THEN 1 ELSE 0 END) AS midrange_count,
        SUM(CASE WHEN price_tier='Premium'   THEN 1 ELSE 0 END) AS premium_count,
        SUM(CASE WHEN price_tier='Luxury'    THEN 1 ELSE 0 END) AS luxury_count
    FROM perfume_prices
    WHERE price_inr IS NOT NULL
    GROUP BY platform
    ORDER BY avg_price_inr
""")

print("=" * 60)
print("BUSINESS SUMMARY — INDIAN PERFUME PRICING INTELLIGENCE")
print("=" * 60)
print(summary.to_string(index=False))
print()
print("KEY INSIGHTS FOR INTERVIEW:")
print("-" * 60)

for _, row in summary.iterrows():
    print(f"""
Platform : {row['platform']}
Listings : {int(row['total_listings'])} products analysed
Avg Price: ₹{int(row['avg_price_inr']):,}
Range    : ₹{int(row['min_price_inr']):,} → ₹{int(row['max_price_inr']):,}
Segments : Budget={int(row['budget_count'])} | Mid={int(row['midrange_count'])} | Premium={int(row['premium_count'])} | Luxury={int(row['luxury_count'])}
""")

print("✅ Full analysis complete. Save this notebook and upload to GitHub.")

BUSINESS SUMMARY — INDIAN PERFUME PRICING INTELLIGENCE
 platform  total_listings  avg_price_inr  min_price_inr  max_price_inr  budget_count  midrange_count  premium_count  luxury_count
 Flipkart              17         759.00         140.00       2,322.00             9               6              2             0
Amazon.in              47       1,669.00         149.00      26,999.00            35               6              2             4
    Nykaa              27       4,707.00         349.00      10,500.00             2              12              3            10

KEY INSIGHTS FOR INTERVIEW:
------------------------------------------------------------

Platform : Flipkart
Listings : 17 products analysed
Avg Price: ₹759
Range    : ₹140 → ₹2,322
Segments : Budget=9 | Mid=6 | Premium=2 | Luxury=0


Platform : Amazon.in
Listings : 47 products analysed
Avg Price: ₹1,669
Range    : ₹149 → ₹26,999
Segments : Budget=35 | Mid=6 | Premium=2 | Luxury=4


Platform : Nykaa
Listings : 27 produc

In [ ]:
# ============================================================
# CELL 12: ANALYSIS 7 — Brand-Level Pricing Intelligence
# Business question: Which brands dominate each platform
# and how does their pricing differ across platforms?
# ============================================================

# STEP 1: Extract brand name from product_name
# Most product names start with the brand — we grab the first word
# Example: "Bella Vita Luxury Perfumes..." → "Bella Vita"

df["brand_clean"] = df["product_name"].str.extract(r"^([A-Za-z]+(?:\s[A-Za-z]+)?)")
# r"^([A-Za-z]+(?:\s[A-Za-z]+)?)" = regex that grabs the first 1-2 words
# ^ = start of string
# [A-Za-z]+ = one or more letters
# (?:\s[A-Za-z]+)? = optionally grab a second word too

# Push updated df back into SQLite
df.to_sql("perfume_prices", conn, if_exists="replace", index=False)

# STEP 2: Top brands by listing count
brand_volume = sql("""
    SELECT
        brand_clean                         AS brand,
        COUNT(*)                            AS total_listings,
        COUNT(DISTINCT platform)            AS platforms_present,
        ROUND(AVG(price_inr), 0)            AS avg_price,
        ROUND(MIN(price_inr), 0)            AS min_price,
        ROUND(MAX(price_inr), 0)            AS max_price
    FROM perfume_prices
    WHERE price_inr IS NOT NULL
      AND brand_clean IS NOT NULL
    GROUP BY brand_clean
    HAVING total_listings >= 2             -- only brands with 2+ listings
    ORDER BY total_listings DESC
    LIMIT 15
""")
# HAVING = filters after GROUP BY (like WHERE but for aggregated results)
# total_listings >= 2 = ignore one-off products, focus on real brands

print("=== TOP 15 BRANDS BY LISTING VOLUME ===")
print(brand_volume.to_string(index=False))

# STEP 3: Brands present on multiple platforms (cross-platform brands)
cross_platform = sql("""
    SELECT
        brand_clean                         AS brand,
        COUNT(DISTINCT platform)            AS platforms,
        GROUP_CONCAT(DISTINCT platform)     AS platform_list,
        ROUND(AVG(price_inr), 0)            AS avg_price
    FROM perfume_prices
    WHERE price_inr IS NOT NULL
      AND brand_clean IS NOT NULL
    GROUP BY brand_clean
    HAVING platforms > 1
    ORDER BY platforms DESC, avg_price DESC
""")
# GROUP_CONCAT = joins values into one string e.g. "Amazon.in,Nykaa"

print("\n=== BRANDS SELLING ON MULTIPLE PLATFORMS ===")
print(cross_platform.to_string(index=False))

# STEP 4: Platform pricing rank per brand (window function)
brand_platform_rank = sql("""
    SELECT
        brand_clean                                         AS brand,
        platform,
        ROUND(AVG(price_inr), 0)                           AS avg_price,
        RANK() OVER (
            PARTITION BY brand_clean
            ORDER BY AVG(price_inr) DESC
        )                                                   AS price_rank
    FROM perfume_prices
    WHERE price_inr IS NOT NULL
      AND brand_clean IS NOT NULL
    GROUP BY brand_clean, platform
    HAVING COUNT(*) >= 2
    ORDER BY brand, price_rank
""")
# RANK() OVER (PARTITION BY brand ORDER BY avg_price DESC)
# = for each brand separately, rank platforms from most expensive to cheapest
# This is a WINDOW FUNCTION — mention this in your interview

print("\n=== PLATFORM PRICE RANK PER BRAND ===")
print(brand_platform_rank.to_string(index=False))

# STEP 5: Visualise top 10 brands by avg price
top_brands = brand_volume.head(10)

fig7 = px.bar(
    top_brands,
    x="brand",
    y="avg_price",
    color="platforms_present",
    text="avg_price",
    title="Top 10 Brands — Avg Price & Platform Reach",
    labels={
        "avg_price": "Avg Price (₹)",
        "brand": "Brand",
        "platforms_present": "# Platforms"
    },
    color_continuous_scale="Viridis"
)
fig7.update_traces(texttemplate="₹%{text:,.0f}", textposition="outside")
fig7.update_layout(plot_bgcolor="white", xaxis_tickangle=-30)
fig7.show()

# STEP 6: Interview-ready brand insight
print("\n" + "=" * 60)
print("BRAND INSIGHTS FOR INTERVIEW")
print("=" * 60)
print(f"Total unique brands identified : {df['brand_clean'].nunique()}")
print(f"Brands on 2+ platforms         : {len(cross_platform)}")
if len(cross_platform) > 0:
    top = brand_volume.iloc[0]
    print(f"Most listed brand              : {top['brand']} ({int(top['total_listings'])} listings)")
    print(f"Highest avg price brand        : {brand_volume.sort_values('avg_price', ascending=False).iloc[0]['brand']}")
print("\n✅ Cell 12 complete. Your analysis is done.")

=== TOP 15 BRANDS BY LISTING VOLUME ===
           brand  total_listings  platforms_present  avg_price  min_price  max_price
      Bella Vita               4                  1     274.00     149.00     549.00
  Buy Fragrances               3                  1   4,399.00     999.00  10,500.00
        Buy Best               3                  1   8,150.00   4,500.00  10,500.00
   Engage Luxury               2                  1     399.00     399.00     399.00
          Buy La               2                  2     799.00     799.00     799.00
Beardo Godfather               2                  1     308.00     159.00     456.00
   Arabian Aroma               2                  1     269.00     268.00     270.00

=== BRANDS SELLING ON MULTIPLE PLATFORMS ===
 brand  platforms  platform_list  avg_price
Buy La          2 Flipkart,Nykaa     799.00

=== PLATFORM PRICE RANK PER BRAND ===
           brand  platform  avg_price  price_rank
   Arabian Aroma Amazon.in     269.00           1
Beardo 


BRAND INSIGHTS FOR INTERVIEW
Total unique brands identified : 133
Brands on 2+ platforms         : 1
Most listed brand              : Bella Vita (4 listings)
Highest avg price brand        : Buy Best

✅ Cell 12 complete. Your analysis is done.
